# 1. Dataset

In [1]:
import numpy as np

# -------------------- Dataset --------------------

# X Shape = (4,2)

X = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
], dtype=float)

# y Shape = (4,1)

y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

print("X Shape :", X.shape)
print("y Shape :", y.shape)

X Shape : (4, 2)
y Shape : (4, 1)


# 2. tensorflow

In [8]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="sigmoid"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(X, y, epochs=100, verbose=0)

loss, accuracy = model.evaluate(X, y, verbose=0)

print("Accuracy :", accuracy)

pred = model.predict(X, verbose=0)

print("\nProbability:")
print(pred)

print("\nPrediction:")
print((pred >= 0.5).astype(int))

print("\nActual:")
print(y)

Accuracy : 0.5

Probability:
[[0.6038685 ]
 [0.5865914 ]
 [0.620797  ]
 [0.60307354]]

Prediction:
[[1]
 [1]
 [1]
 [1]]

Actual:
[[0.]
 [1.]
 [1.]
 [0.]]


# 3. Scratch

In [9]:
class MLP:

    def __init__(self, input_size, hidden_size, output_size,
                 lr=0.1, epochs=100):

        self.lr = lr
        self.epochs = epochs

        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        return x * (1 - x)

    def fit(self, X, y):

        n = len(X)

        for _ in range(self.epochs):

            # ---------- Forward Pass ----------

            self.Z1 = X @ self.W1 + self.b1
            self.A1 = self.sigmoid(self.Z1)

            self.Z2 = self.A1 @ self.W2 + self.b2
            self.A2 = self.sigmoid(self.Z2)

            # ---------- Backpropagation ----------

            dZ2 = self.A2 - y

            dW2 = self.A1.T @ dZ2 / n
            db2 = np.sum(dZ2, axis=0, keepdims=True) / n

            dA1 = dZ2 @ self.W2.T
            dZ1 = dA1 * self.sigmoid_derivative(self.A1)

            dW1 = X.T @ dZ1 / n
            db1 = np.sum(dZ1, axis=0, keepdims=True) / n

            # ---------- Update ----------

            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2

            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1

    def predict(self, X):

        A1 = self.sigmoid(X @ self.W1 + self.b1)
        A2 = self.sigmoid(A1 @ self.W2 + self.b2)

        return (A2 >= 0.5).astype(int)

In [10]:
# -------------------- Train --------------------

model = MLP(
    input_size=2,
    hidden_size=4,
    output_size=1,
    lr=0.1,
    epochs=100
)

model.fit(X, y)

# -------------------- Predict --------------------

pred = model.predict(X)

print("Prediction:")
print(pred)

print("\nActual:")
print(y)

# -------------------- Accuracy --------------------

accuracy = np.mean(pred == y)

print("\nAccuracy:", accuracy)

Prediction:
[[1]
 [1]
 [1]
 [0]]

Actual:
[[0.]
 [1.]
 [1.]
 [0.]]

Accuracy: 0.75


# Multi-Layer Perceptron (MLP) - From Scratch

## Architecture

Input Layer

$$
X
\rightarrow
Hidden
\rightarrow
Output
$$

---

# Step 1 : Hidden Layer

### Linear Transformation

$$
Z_1=XW_1+b_1
$$

Code

```python
self.Z1 = X @ self.W1 + self.b1
```

---

### Activation

$$
A_1=\sigma(Z_1)
$$

where

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

Code

```python
self.A1 = self.sigmoid(self.Z1)
```

---

# Step 2 : Output Layer

### Linear Transformation

$$
Z_2=A_1W_2+b_2
$$

Code

```python
self.Z2 = self.A1 @ self.W2 + self.b2
```

---

### Activation

$$
A_2=\sigma(Z_2)
$$

Code

```python
self.A2 = self.sigmoid(self.Z2)
```

---

# Step 3 : Loss Function

Binary Cross Entropy

$$
J
=
-\frac1n
\sum
\left[
y\log(A_2)
+
(1-y)\log(1-A_2)
\right]
$$

---

# Step 4 : Output Layer Gradient

For BCE + Sigmoid,

$$
dZ_2=A_2-y
$$

Code

```python
dZ2 = self.A2 - y
```

---

### Weight Gradient

$$
dW_2
=
\frac1n
A_1^TdZ_2
$$

Code

```python
dW2 = self.A1.T @ dZ2 / n
```

---

### Bias Gradient

$$
db_2
=
\frac1n
\sum dZ_2
$$

Code

```python
db2 = np.sum(dZ2, axis=0, keepdims=True) / n
```

---

# Step 5 : Hidden Layer Gradient

$$
dA_1
=
dZ_2W_2^T
$$

Code

```python
dA1 = dZ2 @ self.W2.T
```

---

$$
dZ_1
=
dA_1
\odot
\sigma'(A_1)
$$

where

$$
\sigma'(A)
=
A(1-A)
$$

Code

```python
dZ1 = dA1 * self.sigmoid_derivative(self.A1)
```

---

### Weight Gradient

$$
dW_1
=
\frac1n
X^TdZ_1
$$

Code

```python
dW1 = X.T @ dZ1 / n
```

---

### Bias Gradient

$$
db_1
=
\frac1n
\sum dZ_1
$$

Code

```python
db1 = np.sum(dZ1, axis=0, keepdims=True) / n
```

---

# Step 6 : Gradient Descent Update

Output Layer

$$
W_2
=
W_2
-
\alpha
dW_2
$$

$$
b_2
=
b_2
-
\alpha
db_2
$$

Code

```python
self.W2 -= self.lr * dW2
self.b2 -= self.lr * db2
```

---

Hidden Layer

$$
W_1
=
W_1
-
\alpha
dW_1
$$

$$
b_1
=
b_1
-
\alpha
db_1
$$

Code

```python
self.W1 -= self.lr * dW1
self.b1 -= self.lr * db1
```

---

# Step 7 : Prediction

$$
\hat y
=
\begin{cases}
1,&A_2\ge0.5\\
0,&A_2<0.5
\end{cases}
$$

Code

```python
return (A2 >= 0.5).astype(int)
```

---

# Shapes

| Variable | Shape |
|-----------|-------|
| $X$ | $(n,input)$ |
| $W_1$ | $(input,hidden)$ |
| $b_1$ | $(1,hidden)$ |
| $Z_1$ | $(n,hidden)$ |
| $A_1$ | $(n,hidden)$ |
| $W_2$ | $(hidden,output)$ |
| $b_2$ | $(1,output)$ |
| $Z_2$ | $(n,output)$ |
| $A_2$ | $(n,output)$ |
| $dW_2$ | $(hidden,output)$ |
| $db_2$ | $(1,output)$ |
| $dW_1$ | $(input,hidden)$ |
| $db_1$ | $(1,hidden)$ |